In [1]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional

import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
DOMS_DIR = Path("doms1")
OUT_CSV = "parsed_tiktok1.csv"


def extract_video_id_from_filename(path: Path) -> Optional[str]:
    """
    Достает video_id из имени файла.
    Например:
    1_7573978385748069639.html -> 7573978385748069639
    """
    m = re.search(r"(\d{18,20})", path.stem)
    return m.group(1) if m else None


def extract_page_title(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "html.parser")
    return soup.title.get_text(strip=True) if soup.title else None


def find_rehydration_json(html: str) -> dict:
    """
    Ищет основной JSON TikTok внутри script-тега.
    Сначала пробует точный id, потом запасной поиск.
    """
    patterns = [
        r'<script id="__UNIVERSAL_DATA_FOR_REHYDRATION__" type="application/json">(.*?)</script>',
        r"<script[^>]*>(.*?)</script>",
    ]

    m = re.search(patterns[0], html, flags=re.DOTALL)
    if m:
        return json.loads(m.group(1))

    for match in re.finditer(patterns[1], html, flags=re.DOTALL):
        script_text = match.group(1)
        if "webapp.video-detail" in script_text and "itemStruct" in script_text:
            try:
                return json.loads(script_text)
            except json.JSONDecodeError:
                continue

    raise ValueError("Не найден JSON с данными TikTok")


def deep_find_video_obj(obj: Any, target_video_id: str) -> Optional[dict]:
    """
    Рекурсивно ищет объект видео с нужным id.
    """
    if isinstance(obj, dict):
        if str(obj.get("id")) == str(target_video_id) and (
            "stats" in obj or "statsV2" in obj or "desc" in obj or "createTime" in obj
        ):
            return obj

        for value in obj.values():
            found = deep_find_video_obj(value, target_video_id)
            if found:
                return found

    elif isinstance(obj, list):
        for item in obj:
            found = deep_find_video_obj(item, target_video_id)
            if found:
                return found

    return None


def unix_to_date_str(ts: Any) -> Optional[str]:
    if ts is None or ts == "":
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).date().isoformat()
    except Exception:
        return None


def parse_one_file(path: Path) -> dict:
    result = {
        "file_name": path.name,
        "video_id": None,
        "page_title": None,
        "caption": None,
        "create_time_unix": None,
        "create_date_utc": None,
        "likes": None,
        "comments": None,
        "shares": None,
        "status": "ok",
        "error": None,
    }

    try:
        video_id = extract_video_id_from_filename(path)
        if not video_id:
            raise ValueError("Не удалось извлечь video_id из имени файла")

        html = path.read_text(encoding="utf-8")
        page_title = extract_page_title(html)
        data = find_rehydration_json(html)
        video = deep_find_video_obj(data, video_id)

        if not video:
            raise ValueError(f"Не найден объект видео с id={video_id}")

        stats = video.get("stats", {}) or {}
        stats_v2 = video.get("statsV2", {}) or {}

        # Иногда значения лежат в stats, иногда в statsV2
        likes = stats.get("diggCount", stats_v2.get("diggCount"))
        comments = stats.get("commentCount", stats_v2.get("commentCount"))
        shares = stats.get("shareCount", stats_v2.get("shareCount"))
        create_time = video.get("createTime")

        result.update({
            "video_id": str(video.get("id", video_id)),
            "page_title": page_title,
            "caption": video.get("desc"),
            "create_time_unix": create_time,
            "create_date_utc": unix_to_date_str(create_time),
            "likes": int(likes) if likes is not None and str(likes).isdigit() else likes,
            "comments": int(comments) if comments is not None and str(comments).isdigit() else comments,
            "shares": int(shares) if shares is not None and str(shares).isdigit() else shares,
        })

    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)

    return result


def main():
    if not DOMS_DIR.exists():
        raise FileNotFoundError(f"Папка не найдена: {DOMS_DIR}")

    html_files = sorted(DOMS_DIR.glob("*.html"))
    if not html_files:
        raise FileNotFoundError(f"В папке {DOMS_DIR} нет HTML-файлов")

    rows = []
    for i, path in enumerate(html_files, start=1):
        print(f"[{i}/{len(html_files)}] Parsing {path.name}")
        rows.append(parse_one_file(path))

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\nSaved results to: {OUT_CSV}")
    print(df[["file_name", "video_id", "create_date_utc", "likes", "comments", "shares", "status"]].head())


if __name__ == "__main__":
    main()

[1/521] Parsing 100_7609292985380998421.html
[2/521] Parsing 101_7609281412172533012.html
[3/521] Parsing 102_7609139857323085076.html
[4/521] Parsing 103_7606948379381222677.html
[5/521] Parsing 104_7606580013478972693.html
[6/521] Parsing 105_7606566023306857748.html
[7/521] Parsing 106_7606541955593522453.html
[8/521] Parsing 107_7605979341209734421.html
[9/521] Parsing 108_7605904808188415252.html
[10/521] Parsing 109_7605622007367093524.html
[11/521] Parsing 10_7529550720392023303.html
[12/521] Parsing 110_7605613235819154708.html
[13/521] Parsing 111_7605206101189004565.html
[14/521] Parsing 112_7605194154913713429.html
[15/521] Parsing 113_7605183547116162325.html
[16/521] Parsing 114_7604810852797025556.html
[17/521] Parsing 115_7604797582795230485.html
[18/521] Parsing 116_7604488087724428564.html
[19/521] Parsing 117_7604464560237497620.html
[20/521] Parsing 118_7604450387302632725.html
[21/521] Parsing 119_7604397523050024212.html
[22/521] Parsing 11_7529547333021846791.html

In [ ]:
# Files
dataset_file = "dataset.csv"
parsed_file = "parsed_tiktok.csv"
output_file = "dataset_merged.csv"


def extract_video_id_from_url(url):
    """Extract TikTok video ID from a video URL."""
    if pd.isna(url):
        return None
    url = str(url).strip().strip("'").strip('"')
    match = re.search(r"/video/(\d+)", url)
    return match.group(1) if match else None


def extract_video_id_from_filename(filename):
    """Extract TikTok video ID from parsed HTML file name like 100_7566870951661423880.html"""
    if pd.isna(filename):
        return None
    filename = str(filename).strip()
    match = re.search(r"_(\d+)\.html$", filename)
    return match.group(1) if match else None


# Load files
dataset = pd.read_csv(dataset_file)
parsed = pd.read_csv(parsed_file)

# Extract clean IDs
dataset["video_id_key"] = dataset["video_url"].apply(extract_video_id_from_url)
parsed["video_id_key"] = parsed["file_name"].apply(extract_video_id_from_filename)

# Keep only needed columns from parsed
parsed_for_merge = parsed.rename(columns={
    "page_title": "title"
})[
    [
        "video_id_key",
        "title",
        "caption",
        "create_time_unix",
        "create_date_utc",
        "shares",
        "comments",
        "likes",
    ]
]

# Remove duplicate parsed IDs just in case
parsed_for_merge = parsed_for_merge.drop_duplicates(subset="video_id_key")

# Merge
merged = dataset.merge(
    parsed_for_merge,
    on="video_id_key",
    how="left",
    suffixes=("", "_parsed")
)

# Update dataset columns with parsed values
columns_to_update = [
    "title",
    "caption",
    "create_time_unix",
    "create_date_utc",
    "shares",
    "comments",
    "likes",
]

for col in columns_to_update:
    parsed_col = f"{col}_parsed"
    if parsed_col in merged.columns:
        # overwrite with parsed values if available
        merged[col] = merged[parsed_col].combine_first(merged[col])

# Drop helper columns
drop_cols = ["video_id_key"] + [f"{col}_parsed" for col in columns_to_update if f"{col}_parsed" in merged.columns]
merged = merged.drop(columns=drop_cols, errors="ignore")

# Save
merged.to_csv(output_file, index=False)

# Diagnostics
dataset_ids = dataset["video_url"].apply(extract_video_id_from_url)
parsed_ids = parsed["file_name"].apply(extract_video_id_from_filename)

matched = dataset_ids.isin(set(parsed_ids)).sum()
unmatched_dataset = dataset.loc[~dataset_ids.isin(set(parsed_ids)), ["video_url"]].copy()
unmatched_parsed = parsed.loc[~parsed_ids.isin(set(dataset_ids)), ["file_name"]].copy()

print(f"Done. Merged file saved as: {output_file}")
print(f"Rows in dataset: {len(dataset)}")
print(f"Rows in parsed: {len(parsed)}")
print(f"Rows in merged: {len(merged)}")
print(f"Matched dataset rows by ID: {matched}")
print(f"Unmatched dataset rows: {len(unmatched_dataset)}")
print(f"Unmatched parsed rows: {len(unmatched_parsed)}")

if len(unmatched_dataset) > 0:
    print("\nUnmatched dataset URLs:")
    print(unmatched_dataset.to_string(index=False))

if len(unmatched_parsed) > 0:
    print("\nUnmatched parsed files:")
    print(unmatched_parsed.to_string(index=False))

Done. Merged file saved as: dataset_merged.csv
Rows in dataset: 693
Rows in parsed: 693
Rows in merged: 693
Matched dataset rows by ID: 691
Unmatched dataset rows: 2
Unmatched parsed rows: 2

Unmatched dataset URLs:
                                                        video_url
https://www.tiktok.com/@bankerthichcode/photo/7587397655681174805
https://www.tiktok.com/@bankerthichcode/photo/7571076768535170322

Unmatched parsed files:
                                                             file_name
48_https_www_tiktok_com_bankerthichcode_photo_7587397655681174805.html
88_https_www_tiktok_com_bankerthichcode_photo_7571076768535170322.html


In [9]:
input_file = "dataset_merged.csv"
output_file = "dataset_clean_views.csv"

df = pd.read_csv(input_file)


def convert_views(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    match = re.match(r"([\d\.]+)([KMB]?)", value)
    if not match:
        return None

    number = float(match.group(1))
    suffix = match.group(2)

    multiplier = {
        "": 1,
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000
    }

    return int(number * multiplier.get(suffix, 1))


df["num_views"] = df["num_views"].apply(convert_views)

df.to_csv(output_file, index=False)

print("Done. Clean dataset saved as:", output_file)

Done. Clean dataset saved as: dataset_clean_views.csv


In [2]:
DOMS_DIR = Path("doms")
OUT_FILE = "video_durations.csv"


def extract_video_id_from_filename(path: Path) -> str:
    """
    Extract video_id from filenames like:
    1_7573978385748069639.html -> 7573978385748069639
    """
    stem = path.stem  # e.g. "1_7573978385748069639"
    if "_" in stem:
        return stem.split("_", 1)[1]
    return stem


def try_parse_json_blob(text: str):
    """
    Try to find the main TikTok JSON blob and parse it.
    """
    patterns = [
        r'<script id="__UNIVERSAL_DATA_FOR_REHYDRATION__" type="application/json">(.*?)</script>',
        r'<script id="SIGI_STATE" type="application/json">(.*?)</script>',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            raw_json = match.group(1)
            try:
                return json.loads(raw_json)
            except json.JSONDecodeError:
                pass

    return None


def find_nested_value(obj, target_key):
    """
    Recursively find all values for a given key in nested dict/list structures.
    """
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key:
                results.append(value)
            results.extend(find_nested_value(value, target_key))
    elif isinstance(obj, list):
        for item in obj:
            results.extend(find_nested_value(item, target_key))

    return results


def extract_duration_from_html(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="ignore")
    video_id = extract_video_id_from_filename(path)

    data = try_parse_json_blob(text)

    duration = None
    precise_duration = None

    if data is not None:
        # Look for all "duration" keys
        duration_candidates = find_nested_value(data, "duration")

        # Keep only numeric-ish durations
        duration_candidates = [
            x for x in duration_candidates
            if isinstance(x, (int, float)) and 0 < x < 60 * 60
        ]

        if duration_candidates:
            # Usually the first good one is the video duration
            duration = duration_candidates[0]

        # Look for preciseDuration objects or values
        precise_candidates = find_nested_value(data, "preciseDuration")

        flat_precise = []
        for item in precise_candidates:
            if isinstance(item, (int, float)):
                flat_precise.append(item)
            elif isinstance(item, dict):
                if "preciseDuration" in item and isinstance(item["preciseDuration"], (int, float)):
                    flat_precise.append(item["preciseDuration"])

        flat_precise = [x for x in flat_precise if 0 < x < 60 * 60]

        if flat_precise:
            precise_duration = flat_precise[0]

    # Fallback: regex directly on raw HTML
    if duration is None:
        m = re.search(r'"duration":\s*(\d+(?:\.\d+)?)', text)
        if m:
            duration = float(m.group(1))
            if duration.is_integer():
                duration = int(duration)

    if precise_duration is None:
        m = re.search(r'"preciseDuration":\s*{[^}]*"preciseDuration":\s*(\d+(?:\.\d+)?)', text)
        if m:
            precise_duration = float(m.group(1))

    return {
        "video_id": str(video_id),
        "duration": duration,
        "precise_duration": precise_duration,
        "source_file": path.name,
    }


def main():
    rows = []

    html_files = sorted(DOMS_DIR.glob("*.html"))
    print(f"Found {len(html_files)} HTML files")

    for path in html_files:
        try:
            row = extract_duration_from_html(path)
            rows.append(row)
        except Exception as e:
            rows.append({
                "video_id": extract_video_id_from_filename(path),
                "duration": None,
                "precise_duration": None,
                "source_file": path.name,
                "error": str(e),
            })

    df = pd.DataFrame(rows)

    # Optional: keep only useful columns
    cols = ["video_id", "duration", "precise_duration", "source_file"]
    if "error" in df.columns:
        cols.append("error")
    df = df[cols]

    df.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")

    print(f"Done. Saved to {OUT_FILE}")
    print(df.head())


if __name__ == "__main__":
    main()

Found 693 HTML files
Done. Saved to video_durations.csv
              video_id  duration  precise_duration  \
0  7566870951661423880      99.0         99.683250   
1  7609292985380998421     170.0        278.099550   
2  7566870272913984776      90.0         90.984436   
3  7609281412172533012     104.0        278.099550   
4  7565173148216347922     222.0        222.040820   

                    source_file  
0  100_7566870951661423880.html  
1  100_7609292985380998421.html  
2  101_7566870272913984776.html  
3  101_7609281412172533012.html  
4  102_7565173148216347922.html  


In [3]:
# Load files
df_main = pd.read_csv("dataset_clean_views.csv")
df_dur = pd.read_csv("video_durations.csv", dtype={"video_id": str})

# Extract video_id from video_url
df_main["video_id"] = df_main["video_url"].astype(str).str.extract(r"/video/(\d+)")

# Merge: keep all rows from dataset_clean_views, add duration from video_durations
merged = df_main.merge(
    df_dur[["video_id", "duration"]],
    on="video_id",
    how="left"
)

# Save result
merged.to_csv("dataset_clean_views_with_duration.csv", index=False)

print("Done. Saved as dataset_clean_views_with_duration.csv")
print("Rows:", len(merged))
print("Matched durations:", merged["duration"].notna().sum())

Done. Saved as dataset_clean_views_with_duration.csv
Rows: 693
Matched durations: 687
